In [1]:
import numpy as np

# 학생 4명 × 과목 2개 → 4행 2열(4×2) 행렬
scores = np.array([
    [85, 90],   # 철수
    [92, 88],   # 영희
    [70, 85],   # 민수
    [60, 75]    # 지수
])

print("shape(행, 열):", scores.shape)  # (4, 2) → 행 먼저, 열 나중
print("차원 수(ndim) :", scores.ndim)   # 2 (행렬은 2차원)
print("전체 원소 수  :", scores.size)   # 8 (=4×2)

shape(행, 열): (4, 2)
차원 수(ndim) : 2
전체 원소 수  : 8


In [3]:
import numpy as np

# 5×5 흑백 이미지 (255=흰색, 0=검정) — 가운데에 십자 모양이 그려진 그림
image = np.array([
    [  0,   0, 255,   0,   0],
    [  0,   0, 255,   0,   0],
    [255, 255, 255, 255, 255],
    [  0,   0, 255,   0,   0],
    [  0,   0, 255,   0,   0]
])

print("이미지 크기(shape):", image.shape)  # (5, 5)
# 밝은 칸(255)은 ■, 어두운 칸(0)은 · 로 그려보기
# "구분자".join(문자열_리스트_또는_생성자)
for row in image:
    print("".join("■" if v > 127 else "·" for v in row))

이미지 크기(shape): (5, 5)
··■··
··■··
■■■■■
··■··
··■··


In [4]:
import numpy as np

A = np.array([[1, 2], [3, 4]])  # (m=2, n=2)
B = np.array([[5, 6], [7, 8]])  # (n=2, p=2)

m, n = A.shape          # A: m행 n열
n2, p = B.shape         # B: n행 p열
assert n == n2, "내부 차원(n)이 안 맞으면 곱할 수 없음!"

C = np.zeros((m, p))    # 결과는 m×p 크기
for i in range(m):           # 결과의 행
    for j in range(p):       # 결과의 열
        for k in range(n):   # 짝지어 곱할 대상(합의 인덱스)
            C[i, j] += A[i, k] * B[k, j]   # 곱해서 누적 = 내적

print("직접 구현한 결과:\n", C)
print("NumPy @ 결과:\n", A @ B)

직접 구현한 결과:
 [[19. 22.]
 [43. 50.]]
NumPy @ 결과:
 [[19 22]
 [43 50]]


In [5]:
import numpy as np

# 풀고 싶은 방정식:  A x = b
A = np.array([[2., 1.],
               [1., 3.]])
b = np.array([5., 10.])

# 방법 1) 역행렬을 직접 구해서 곱하기: x = A⁻¹ b
x1 = np.linalg.inv(A) @ b

# 방법 2) 전용 solver 사용 (실무 권장) — 내부적으로 더 안정적/빠름
x2 = np.linalg.solve(A, b)

print("역행렬로 푼 x :", x1)
print("solve로 푼 x  :", x2)
print("검산 A@x = b ?:", np.allclose(A @ x2, b))

# LinAlgError: Singular matrix → 그 행렬은 역행렬이 없습니다(행렬식 0). 데이터가 겹치거나 종속인 경우 자주 발생
# dtype이 정수(int)면 소수점 결과가 잘려 엉뚱한 값이 나올 수 있으니 2., 1.처럼 실수(float)로 만들 것

역행렬로 푼 x : [1. 3.]
solve로 푼 x  : [1. 3.]
검산 A@x = b ?: True


In [6]:
import numpy as np

# ── 데이터: 5명의 키(cm)와 몸무게(kg) ──────────────────────
height = np.array([165, 170, 175, 180, 185], dtype=float)
weight = np.array([58,  65,  70,  75,  82],  dtype=float)

# ── 설계행렬 A 만들기 ─────────────────────────────────────
# 찾고 싶은 직선: weight = a×height + b×1
# → 각 행이 (height, 1)이 되도록 1로 채운 열을 붙입니다 (절편 항!)
A = np.column_stack([height, np.ones(len(height))])
b = weight

print("A shape:", A.shape, "| b shape:", b.shape)   # (5,2) (5,)

# ── 방법 1) 정규방정식 공식을 그대로 구현 (이해용) ──────────ㅁ
# x̂ = (AᵀA)⁻¹ Aᵀb  ← 방금 유도한 공식 그 자체
x_formula = np.linalg.inv(A.T @ A) @ A.T @ b

# ── 방법 2) lstsq 사용 (실무 권장) ────────────────────────
# rcond=None: 버전 경고 방지 / [0]: 반환 튜플의 첫 원소가 해
x_lstsq = np.linalg.lstsq(A, b, rcond=None)[0]

a_hat, b_hat = x_lstsq          # 튜플 언패킹: 기울기, 절편
print("공식 방식 :", np.round(x_formula, 4))
print("lstsq     :", np.round(x_lstsq,   4))
print(f"찾은 직선: 몸무게 = {a_hat:.2f} × 키 + ({b_hat:.2f})")

# ── 잔차와 오차 제곱합 확인 ───────────────────────────────
pred     = A @ x_lstsq          # 예측값 ŷ = Ax̂
residual = b - pred             # 잔차 e = b - Ax̂
sse      = np.sum(residual ** 2)   # 오차 제곱합 S = Σeᵢ²

print("잔차      :", np.round(residual, 3))
print("잔차 합   :", round(residual.sum(), 10), "← 0이어야 정상")
print("오차제곱합:", round(sse, 4))

# ── 직교 조건 검증: Aᵀe = 0 이어야 함 (유도 ①단계) ─────────
print("Aᵀe (≈0?) :", np.round(A.T @ residual, 10))

# ── 예측해보기 ────────────────────────────────────────────
new_height = 168
print(f"키 {new_height}cm 예측 몸무게: {a_hat * new_height + b_hat:.1f}kg")

A shape: (5, 2) | b shape: (5,)
공식 방식 : [   1.16 -133.  ]
lstsq     : [   1.16 -133.  ]
찾은 직선: 몸무게 = 1.16 × 키 + (-133.00)
잔차      : [-0.4  0.8  0.  -0.8  0.4]
잔차 합   : 0.0 ← 0이어야 정상
오차제곱합: 1.6
Aᵀe (≈0?) : [1.e-10 0.e+00]
키 168cm 예측 몸무게: 61.9kg
